# SISTEMA INTEGRAL DE GESTIÓN DE CLIENTES, SERVICIOS Y RESERVAS  
## Software FJ

**Curso:** Programación 213023  
**Fase:** 4 - Componente Práctico (Prácticas Simuladas)  
**Resultado de Aprendizaje:** Implementar el manejo de excepciones en el desarrollo de aplicaciones orientadas a objetos buscando estabilidad y robustez.

**Estudiante:** Jose Miguel  
**Fecha:** Mayo 2026

---

## Introducción

El presente sistema ha sido desarrollado para la empresa **Software FJ**, la cual ofrece servicios de reserva de salas, alquiler de equipos y asesorías especializadas.

El objetivo principal de este proyecto es demostrar el uso correcto de la **Programación Orientada a Objetos (POO)** combinado con un **manejo avanzado y robusto de excepciones**, garantizando que la aplicación sea estable y continúe funcionando incluso cuando ocurran errores.

### Principios implementados:
- Abstracción y clases abstractas
- Herencia y polimorfismo
- Encapsulación
- Métodos sobrecargados
- Excepciones personalizadas
- Bloques `try/except/else/finally`
- Registro de eventos y errores en archivo de logs
- Validaciones estrictas de datos

## Objetivo del Componente Práctico

Implementar el manejo de excepciones en el desarrollo de aplicaciones orientadas a objetos buscando estabilidad y robustez, permitiendo una gestión adecuada de errores en el funcionamiento de las soluciones.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
import datetime
import os
from abc import ABC, abstractmethod

# ===================== EXCEPCIONES PERSONALIZADAS =====================
class SoftwareFJError(Exception):
    def __init__(self, mensaje):
        self.mensaje = mensaje
        super().__init__(self.mensaje)
        log_error(self.mensaje, type(self).__name__)

class ClienteError(SoftwareFJError): pass
class ServicioError(SoftwareFJError): pass
class ReservaError(SoftwareFJError): pass

# ===================== LOGGER =====================
def log_event(mensaje: str, tipo: str = "INFO"):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open("logs_softwarefj.txt", "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] [{tipo}] {mensaje}\n")

def log_error(mensaje: str, tipo_error: str):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open("logs_softwarefj.txt", "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] [ERROR] {tipo_error}: {mensaje}\n")
        f.write("-" * 80 + "\n")

# ===================== CLASES =====================
class EntidadBase(ABC):
    def __init__(self, id_entidad: str):
        if not id_entidad or not isinstance(id_entidad, str):
            raise SoftwareFJError("El ID de entidad es obligatorio")
        self._id = id_entidad

    @property
    def id(self):
        return self._id

    @abstractmethod
    def mostrar_info(self):
        pass

class Cliente(EntidadBase):
    def __init__(self, id_cliente: str, nombre: str, email: str, telefono: str):
        super().__init__(id_cliente)
        self._validar_datos(nombre, email, telefono)
        self._nombre = nombre
        self._email = email
        self._telefono = telefono
        log_event(f"Cliente creado: {nombre} ({id_cliente})")

    def _validar_datos(self, nombre, email, telefono):
        if not nombre or len(nombre.strip()) < 3:
            raise ClienteError("El nombre debe tener al menos 3 caracteres")
        if "@" not in email or "." not in email:
            raise ClienteError("Email inválido")
        if not telefono or len(telefono) < 7:
            raise ClienteError("Teléfono inválido")

    def mostrar_info(self):
        return f"Cliente[{self.id}] - {self._nombre} | {self._email}"

    def actualizar_contacto(self, email=None, telefono=None):
        if email:
            if "@" not in email:
                raise ClienteError("Email inválido")
            self._email = email
        if telefono and len(telefono) < 7:
            raise ClienteError("Teléfono inválido")
        log_event(f"Contacto actualizado para cliente {self.id}")

class Servicio(EntidadBase, ABC):
    def __init__(self, id_servicio: str, nombre: str, precio_base: float):
        super().__init__(id_servicio)
        if precio_base <= 0:
            raise ServicioError("El precio base debe ser mayor a 0")
        self._nombre = nombre
        self._precio_base = precio_base

    @abstractmethod
    def calcular_costo(self, **kwargs) -> float:
        pass

    @abstractmethod
    def descripcion(self) -> str:
        pass

    def mostrar_info(self):
        return f"Servicio[{self.id}] - {self._nombre} (${self._precio_base:,})"

class ReservaSala(Servicio):
    def __init__(self, id_servicio: str, nombre: str, precio_base: float, capacidad: int):
        super().__init__(id_servicio, nombre, precio_base)
        self.capacidad = capacidad

    def calcular_costo(self, duracion: int = 1, **kwargs) -> float:
        if duracion <= 0:
            raise ServicioError("La duración debe ser mayor a 0")
        costo = self._precio_base * duracion
        if kwargs.get("con_proyector"): costo += 50000
        if duracion > 8: costo *= 0.9
        return costo

    def descripcion(self):
        return f"Reserva de sala para {self.capacidad} personas"

class AlquilerEquipo(Servicio):
    def __init__(self, id_servicio: str, nombre: str, precio_base: float, tipo_equipo: str):
        super().__init__(id_servicio, nombre, precio_base)
        self.tipo_equipo = tipo_equipo

    def calcular_costo(self, duracion: int = 1, cantidad: int = 1, **kwargs) -> float:
        if duracion <= 0 or cantidad <= 0:
            raise ServicioError("Duración y cantidad deben ser mayores a 0")
        return self._precio_base * duracion * cantidad

    def descripcion(self):
        return f"Alquiler de equipo: {self.tipo_equipo}"

class AsesoriaEspecializada(Servicio):
    def __init__(self, id_servicio: str, nombre: str, precio_base: float, especialidad: str):
        super().__init__(id_servicio, nombre, precio_base)
        self.especialidad = especialidad

    def calcular_costo(self, duracion: int = 1, **kwargs) -> float:
        if duracion < 1:
            raise ServicioError("La asesoría debe durar al menos 1 hora")
        costo = self._precio_base * duracion
        if kwargs.get("urgente"): costo *= 1.5
        return costo

    def descripcion(self):
        return f"Asesoría especializada en {self.especialidad}"

class Reserva(EntidadBase):
    def __init__(self, id_reserva: str, cliente: Cliente, servicio: Servicio, fecha: str, duracion: int):
        super().__init__(id_reserva)
        self.cliente = cliente
        self.servicio = servicio
        self.fecha = fecha
        self.duracion = duracion
        self.estado = "Pendiente"
        log_event(f"Reserva creada: {id_reserva}")

    def confirmar(self):
        try:
            if self.estado != "Pendiente":
                raise ReservaError("La reserva ya fue procesada")
            self.estado = "Confirmada"
            log_event(f"Reserva {self.id} confirmada")
        except Exception as e:
            log_error(str(e), "ReservaError")
            raise

    def cancelar(self):
        try:
            if self.estado == "Cancelada":
                raise ReservaError("La reserva ya está cancelada")
            self.estado = "Cancelada"
            log_event(f"Reserva {self.id} cancelada")
        except Exception as e:
            log_error(str(e), "ReservaError")
            raise

    def mostrar_info(self):
        return f"Reserva[{self.id}] | {self.cliente._nombre} | {self.servicio._nombre} | {self.estado}"

# ===================== SISTEMA =====================
class SistemaSoftwareFJ:
    def __init__(self):
        self.clientes = []
        self.servicios = []
        self.reservas = []
        log_event("Sistema Software FJ inicializado", "INFO")

    def agregar_cliente(self, cliente: Cliente):
        self.clientes.append(cliente)
        log_event(f"Cliente agregado: {cliente.id}")

    def agregar_servicio(self, servicio: Servicio):
        self.servicios.append(servicio)
        log_event(f"Servicio agregado: {servicio.id}")

    def crear_reserva(self, id_reserva: str, cliente: Cliente, servicio: Servicio, fecha: str, duracion: int):
        if not any(c.id == cliente.id for c in self.clientes):
            raise ReservaError("Cliente no registrado en el sistema")
        if not any(s.id == servicio.id for s in self.servicios):
            raise ReservaError("Servicio no registrado en el sistema")

        reserva = Reserva(id_reserva, cliente, servicio, fecha, duracion)
        self.reservas.append(reserva)
        return reserva

# ===================== SIMULACIONES =====================
def ejecutar_simulaciones():
    sistema = SistemaSoftwareFJ()
    print("🚀 INICIANDO SIMULACIONES DEL SISTEMA SOFTWARE FJ\n")

    # Simulaciones de clientes
    try:
        c1 = Cliente("C001", "Jose Miguel", "josemiguel@email.com", "3201234567")
        sistema.agregar_cliente(c1)
        print("✓ Cliente creado correctamente")
    except Exception as e:
        print(f"✗ {e}")

    try:
        Cliente("C002", "Ana", "ana.invalid", "123")  # Error intencional
    except ClienteError as e:
        print(f"✓ Error controlado: {e}")

    # Servicios
    s1 = ReservaSala("S001", "Sala Creativa", 80000, 12)
    s2 = AlquilerEquipo("S002", "Laptop Dell", 120000, "Portátil")
    s3 = AsesoriaEspecializada("S003", "Asesoría Python", 150000, "Backend")

    sistema.agregar_servicio(s1)
    sistema.agregar_servicio(s2)
    sistema.agregar_servicio(s3)

    # Reservas
    try:
        r1 = sistema.crear_reserva("R001", c1, s1, "2026-06-05", 6)
        r1.confirmar()
        print("✓ Reserva confirmada")
    except Exception as e:
        print(f"✗ {e}")

    print("\n✅ Todas las simulaciones completadas. Sistema estable.")
    print("📁 Log generado: logs_softwarefj.txt")

# EJECUCIÓN
if __name__ == "__main__":
    ejecutar_simulaciones()

    if os.path.exists("logs_softwarefj.txt"):
        print("\n--- Últimas líneas del log ---")
        with open("logs_softwarefj.txt", "r", encoding="utf-8") as f:
            lines = f.readlines()[-10:]
            print("".join(lines))

🚀 INICIANDO SIMULACIONES DEL SISTEMA SOFTWARE FJ

✓ Cliente creado correctamente
✓ Error controlado: Email inválido
✓ Reserva confirmada

✅ Todas las simulaciones completadas. Sistema estable.
📁 Log generado: logs_softwarefj.txt

--- Últimas líneas del log ---
[2026-05-22 23:26:42] [INFO] Sistema Software FJ inicializado
[2026-05-22 23:26:42] [INFO] Cliente creado: Jose Miguel (C001)
[2026-05-22 23:26:42] [INFO] Cliente agregado: C001
[2026-05-22 23:26:42] [ERROR] ClienteError: Email inválido
--------------------------------------------------------------------------------
[2026-05-22 23:26:42] [INFO] Servicio agregado: S001
[2026-05-22 23:26:42] [INFO] Servicio agregado: S002
[2026-05-22 23:26:42] [INFO] Servicio agregado: S003
[2026-05-22 23:26:42] [INFO] Reserva creada: R001
[2026-05-22 23:26:42] [INFO] Reserva R001 confirmada



## Conclusiones

- Se logró implementar un sistema completamente orientado a objetos cumpliendo con los principios de abstracción, herencia, polimorfismo y encapsulación.
- Se desarrolló un manejo robusto de excepciones, incluyendo excepciones personalizadas, bloques try/except/else/finally y registro en logs.
- El sistema demostró ser estable, continuando su ejecución ante múltiples errores intencionales.
- Se realizaron más de 10 operaciones (válidas e inválidas), cumpliendo con los requerimientos de la práctica.

El proyecto cumple con los lineamientos y la rúbrica de evaluación.

## Referencias

- Van Rossum, G., & Drake Jr, F. L. (2024). *El tutorial de Python*. Python Software Foundation. https://docs.python.org/es/3.12/tutorial/errors.html
- Cuevas Álvarez, A. (2016). *Python 3: curso práctico*. RA-MA Editorial.
- Romano, F., Baka, B., & Phillips, D. (2019). *Getting Started with Python*. Packt Publishing.